# Calculate the difference between the Taunton-Worcester scenario and base

Gather relevant columns.

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
dir_data = 'data'
dir_output = 'output'
# Scenario results:
file_base_geo = 'results_geo_outcomes_base'
file_base_results = 'results_full_base'
file_taunworc_geo = 'results_geo_outcomes_taunworc'
file_taunworc_results = 'results_full_taunworc'
# Output:
file_results_comparison = 'results_full_comparison_taunworc_base'

## Load data

### Scenario results

In [3]:
def load_data(s_bias):
    p = os.path.join(dir_output, file_base_geo + s_bias + '.csv')
    p2 = os.path.join(dir_output, file_base_results + s_bias + '.csv')
    df_base = pd.concat(
        (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),
        axis='columns'
    )
    
    p = os.path.join(dir_output, file_taunworc_geo + s_bias + '.csv')
    p2 = os.path.join(dir_output, file_taunworc_results + s_bias + '.csv')
    df_taunworc = pd.concat(
        (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),
        axis='columns'
    )
    return df_base, df_taunworc

## Gather data

In [4]:
def gather_data(df_base, df_taunworc):
    cols_same = ['area_lower', 'Admissions_2025_Est']
    
    cols_to_gather = [
        'nearest_ivt_unit',
        'transfer_unit',
        'transfer_required',
        'drip_ship_mt_travel_time',
        'admissions_mt',
        'transfers_mt',
        'admissions_mt_transferloss0!67',
        'transfers_mt_transferloss0!67',
        'admissions_weighted_drip_ship_mrs_0-2',
        'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
        'admissions_weighted_drip_ship_mrs_0-2_transferloss0!67',
        'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67',
        'drip_ship_mrs_0-2',
        'lvo_drip_ship_ivt_mt_mrs_0-2',
        'drip_ship_mrs_0-2_transferloss0!67',
        'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67'
    ]
    
    rename_dict_base = dict([(c, f'{c}_base') for c in cols_to_gather])
    rename_dict_taunworc = dict([(c, f'{c}_taunworc') for c in cols_to_gather])
    
    df = pd.concat(
        (df_base[cols_same],
         df_base[cols_to_gather].rename(columns=rename_dict_base),
         df_taunworc[cols_to_gather].rename(columns=rename_dict_taunworc)),
        axis='columns'
    )
    return df

Calculate changes from base:

In [5]:
def calculate_changes(df):
    cols_for_change = [
        'drip_ship_mt_travel_time', 
        'admissions_mt',
        'transfers_mt',
        'admissions_mt_transferloss0!67',
        'transfers_mt_transferloss0!67',
        'admissions_weighted_drip_ship_mrs_0-2',
        'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
        'admissions_weighted_drip_ship_mrs_0-2_transferloss0!67',
        'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67',
        'drip_ship_mrs_0-2',
        'lvo_drip_ship_ivt_mt_mrs_0-2',
        'drip_ship_mrs_0-2_transferloss0!67',
        'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67'
    ]
    
    for col in cols_for_change:
        df[f'change_in_{col}'] = df[f'{col}_taunworc'] - df[f'{col}_base']
        r = 1 if 'time' in col else 5
        df[f'change_in_{col}'] = df[f'change_in_{col}'].round(r)
    
    df['changed'] = (
        (df['nearest_ivt_unit_base'] != df['nearest_ivt_unit_taunworc']) |
        (df['transfer_unit_base'] != df['transfer_unit_taunworc'])
    )
    # df['change_in_lvo_drip_ship_ivt_mt_mrs_0-2'] != 0
    return df

## Run for all scenarios:

In [6]:
for s_bias in ['', '_bias10min']:
    df_base, df_taunworc = load_data(s_bias)
    df = gather_data(df_base, df_taunworc)
    df = calculate_changes(df)

    # Save to file:
    p = os.path.join(dir_output, file_results_comparison + s_bias + '.csv')
    df.to_csv(p)